# Customer Segmentation Using K-Means

**Internship Task 2** · Exploratory analysis, cluster selection, customer profiling, and visualisation.

This notebook uses the supplied `data/Mall_Customers.csv` dataset. The model clusters customers using **Annual Income (k$)** and **Spending Score (1–100)**. `CustomerID`, `Gender`, and `Age` are kept for identification and descriptive profiling only.

> **Dataset scope:** The file does not include transaction-level purchase history. Spending score is used as a proxy for customer spending behavior; this project does not cluster individual purchases or predict future purchase behavior.


## 1. Import libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
FEATURES = ["Annual Income (k$)", "Spending Score (1-100)"]
DATA_PATH = Path("data") / "Mall_Customers.csv"


## 2. Load and validate the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\nMissing values per column:")
display(df.isna().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["CustomerID"].duplicated().sum())
display(df.describe(include="all").T)

required = ["CustomerID", "Gender", "Age", *FEATURES]
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f"Dataset is missing columns: {missing}")
if df[FEATURES].isna().any().any():
    raise ValueError("Clean missing clustering-feature values before fitting K-means.")


## 3. Explore the two clustering features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df[FEATURES[0]], bins=14, edgecolor="white")
axes[0].set(title="Annual income distribution", xlabel="Annual income (k$)", ylabel="Customers")
axes[1].hist(df[FEATURES[1]], bins=14, edgecolor="white")
axes[1].set(title="Spending score distribution", xlabel="Spending score (1–100)", ylabel="Customers")
for ax in axes:
    ax.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.show()

# Gender and age are useful for describing the resulting groups but are not fed into K-means.


## 4. Standardize features and compare cluster counts

In [ ]:
X = df[FEATURES].astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

candidate_rows = []
for k in range(2, 11):
    candidate = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = candidate.fit_predict(X_scaled)
    candidate_rows.append({
        "k": k,
        "inertia": candidate.inertia_,
        "silhouette_score": silhouette_score(X_scaled, labels),
    })
score_df = pd.DataFrame(candidate_rows)
display(score_df.round(4))
selected_k = int(score_df.loc[score_df["silhouette_score"].idxmax(), "k"])
print(f"Selected k = {selected_k}, the highest silhouette score within the tested range 2–10.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(score_df["k"], score_df["inertia"], marker="o")
axes[0].axvline(selected_k, color="red", linestyle="--", label=f"Selected k={selected_k}")
axes[0].set(title="Elbow method", xlabel="Number of clusters (k)", ylabel="Inertia")
axes[0].legend(frameon=False)
axes[1].plot(score_df["k"], score_df["silhouette_score"], marker="o")
axes[1].scatter([selected_k], [score_df.loc[score_df["k"] == selected_k, "silhouette_score"].iloc[0]], s=90, color="red")
axes[1].set(title="Silhouette score", xlabel="Number of clusters (k)", ylabel="Silhouette score")
for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()


## 5. Fit the final K-means model and profile segments

In [ ]:
def segment_name(income: float, spending: float) -> str:
    if income >= 75 and spending >= 65:
        return "High-income, high-spending"
    if income >= 75 and spending < 35:
        return "High-income, low-spending"
    if income <= 40 and spending >= 65:
        return "Lower-income, high-spending"
    if income <= 40 and spending < 35:
        return "Lower-income, low-spending"
    return "Mid-range, moderate-spending"

model = KMeans(n_clusters=selected_k, n_init=20, random_state=RANDOM_STATE)
cluster_id = model.fit_predict(X_scaled)
centers = scaler.inverse_transform(model.cluster_centers_)
cluster_to_segment = {i: segment_name(float(c[0]), float(c[1])) for i, c in enumerate(centers)}
clustered = df.copy()
clustered["Cluster"] = cluster_id
clustered["Segment"] = clustered["Cluster"].map(cluster_to_segment)

profiles = clustered.groupby(["Cluster", "Segment"], as_index=False).agg(
    Customer_Count=("CustomerID", "count"),
    Average_Age=("Age", "mean"),
    Female_Percent=("Gender", lambda s: 100 * (s.astype(str).str.lower() == "female").mean()),
    Average_Income_kUSD=("Annual Income (k$)", "mean"),
    Average_Spending_Score=("Spending Score (1-100)", "mean"),
)
profiles = profiles.round(2)
display(profiles)


## 6. Visualize clusters

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for segment in clustered["Segment"].drop_duplicates():
    group = clustered[clustered["Segment"] == segment]
    ax.scatter(group[FEATURES[0]], group[FEATURES[1]], s=50, alpha=.8, label=segment)
ax.set(title="Customer segments by annual income and spending score", xlabel="Annual income (k$)", ylabel="Spending score (1–100)")
ax.grid(alpha=.2)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

cluster_counts = clustered["Segment"].value_counts()
display(cluster_counts.rename_axis("Segment").to_frame("Customers"))


## 7. Assign a segment to a sample customer

In [ ]:
# Example inputs: annual income = $60,000 (entered as 60 k$), spending score = 50.
sample_customer = pd.DataFrame([[60, 50]], columns=FEATURES)
sample_scaled = scaler.transform(sample_customer)
sample_cluster = int(model.predict(sample_scaled)[0])
print("Cluster ID:", sample_cluster)
print("Assigned segment:", cluster_to_segment[sample_cluster])


## 8. Interpretation and limitations

- K-means assigns each customer to the closest cluster center in standardized feature space.
- **Inertia** measures the within-cluster squared-distance sum and generally declines as the number of clusters grows.
- **Silhouette score** summarizes cohesion and separation; it is not model accuracy and should be considered with the plots and practical context.
- Segment labels are descriptive summaries of cluster centers, not verified behavioral personas.
- The supplied data contains a spending score rather than transaction-level history. More realistic business segmentation could use recency, frequency, monetary value (RFM), basket details, and product-category preferences.

To reproduce the full saved artifacts and run the dashboard, return to the project root and run `python train_model.py`, followed by `streamlit run app.py`.
